# Báo cáo phân tích kết quả thi THPT

## Hai câu hỏi nghiên cứu từ các biểu đồ hiện có trong ứng dụng

Báo cáo tập trung vào hai vấn đề có thể trả lời trực tiếp bằng các biểu đồ của dashboard:

1. **Sự thay đổi điểm trung bình toàn quốc đến từ toàn bộ môn thi hay chỉ tập trung ở một số môn?**
2. **Chênh lệch vùng miền đến từ toàn bộ địa phương hay bị chi phối bởi một số tỉnh, thành?**

Phân tích sử dụng đúng các chỉ số và logic dữ liệu của ứng dụng. Không bổ sung thước đo chuẩn hóa, không gộp vùng theo Bắc–Trung–Nam và không tạo thêm loại biểu đồ ngoài dashboard.

## Phạm vi, nguồn dữ liệu và cách đọc biểu đồ

### Nguồn dữ liệu

Notebook đọc trực tiếp bộ dữ liệu đã được dashboard sinh ra tại **frontend/src/data/data.ts**. Vì vậy, các kết quả dưới đây có thể đối chiếu với số liệu hiển thị trên ứng dụng.

### Phạm vi phân tích

| Câu hỏi | Bộ lọc tương ứng trên ứng dụng | Biểu đồ sử dụng |
|---|---|---|
| Câu hỏi 1 | CT2006, giai đoạn 2022–2024 | Tab **Tổng quan**: điểm TB toàn quốc theo năm; tab **Xu hướng & Môn học**: xu hướng điểm TB theo môn |
| Câu hỏi 2 | Năm 2026, môn **Tổng hợp**, vùng **Toàn quốc** | Tab **Địa phương & Vùng miền**: điểm TB theo vùng, Top/Bottom tỉnh thành và bảng xếp hạng |

### Nguyên tắc diễn giải

- Câu hỏi 1 giữ riêng CT2006 trong giai đoạn 2022–2024 để không trộn biến động điểm với thay đổi chương trình thi.
- Trong logic của dashboard, điểm trung bình toàn quốc là trung bình của các điểm trung bình môn đủ điều kiện trong từng năm.
- Câu hỏi 2 tái tạo đúng cách dashboard tạo điểm **Tổng hợp**: trung bình các điểm trung bình môn của từng tỉnh; sau đó so sánh với trung bình các điểm trung bình môn của vùng.
- Các kết quả mô tả mối liên hệ trong dữ liệu, không được diễn giải như quan hệ nhân quả.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# Đọc đúng nguồn dữ liệu mà dashboard đang sử dụng.
APP_DATA_PATH = Path("../frontend/src/data/data.ts")
raw_text = APP_DATA_PATH.read_text(encoding="utf-8")
json_start = raw_text.index("const data = ") + len("const data = ")
json_end = raw_text.rfind("};") + 1
app_data = json.loads(raw_text[json_start:json_end])

print(f"Đã nạp dữ liệu dashboard từ: {APP_DATA_PATH}")
print(f"Giai đoạn dữ liệu: {app_data['YEARS'][0]}–{app_data['YEARS'][-1]}")

Đã nạp dữ liệu dashboard từ: ../frontend/src/data/data.ts
Giai đoạn dữ liệu: 2022–2026


## Câu hỏi 1 — Điểm trung bình quốc gia thay đổi do toàn bộ môn hay một số môn?

### Câu hỏi nghiên cứu

Khi điểm trung bình toàn quốc tăng trong giai đoạn 2022–2024, mức tăng đó có xuất hiện đồng đều ở các môn hay chủ yếu được tạo ra bởi một số môn học?

### Vì sao câu hỏi này quan trọng?

Điểm trung bình toàn quốc là một chỉ số tổng hợp. Một xu hướng tăng có thể tạo cảm giác rằng chất lượng kết quả thi được cải thiện đồng đều, nhưng chỉ số tổng hợp không cho biết môn nào thực sự thay đổi và môn nào đứng yên hoặc giảm.

### Cách phân tích

1. Đọc đường xu hướng điểm trung bình toàn quốc trong tab **Tổng quan**.
2. Đối chiếu đường xu hướng điểm trung bình của từng môn trong tab **Xu hướng & Môn học**.
3. Tính thay đổi của từng môn từ năm 2022 đến năm 2024.
4. Kiểm tra xem thay đổi trung bình của 9 môn có khớp với thay đổi của chỉ số toàn quốc trong dashboard hay không.

In [ ]:
# Phân tích câu hỏi 1: CT2006, giai đoạn 2022–2024.
# Giữ cùng chương trình để tránh trộn thay đổi mặt bằng điểm với thay đổi chương trình thi.
subject_year = pd.DataFrame(app_data["subjectYearMatrix"])
subject_year = subject_year[
    subject_year["program"].eq("CT2006")
    & subject_year["year"].isin([2022, 2023, 2024])
].copy()

subject_change = (
    subject_year
    .pivot(index="subjectName", columns="year", values="average")
    .rename_axis(None, axis=1)
    .rename(columns={2022: "Điểm TB 2022", 2024: "Điểm TB 2024"})
)
subject_change["Mức thay đổi"] = subject_change["Điểm TB 2024"] - subject_change["Điểm TB 2022"]
subject_change["Chiều thay đổi"] = np.where(subject_change["Mức thay đổi"] > 0, "Tăng", "Giảm/không tăng")
subject_change = subject_change.sort_values("Mức thay đổi", ascending=False)

national_change = (
    pd.DataFrame(app_data["nationalAverageByYear"])
    .query("program == 'CT2006' and year in [2022, 2023, 2024]")
    .set_index("year")["value"]
)
national_delta = float(national_change.loc[2024] - national_change.loc[2022])
mean_subject_delta = float(subject_change["Mức thay đổi"].mean())

assert np.isclose(national_delta, mean_subject_delta, atol=0.01)
assert int((subject_change["Mức thay đổi"] > 0).sum()) == 6
assert int((subject_change["Mức thay đổi"] <= 0).sum()) == 3

print("Điểm TB toàn quốc:", {int(year): round(value, 2) for year, value in national_change.items()})
print(f"Thay đổi toàn quốc 2022→2024: {national_delta:+.2f} điểm")
print(f"Mức thay đổi trung bình của 9 môn: {mean_subject_delta:+.2f} điểm")
display(subject_change.round(2))

Điểm TB toàn quốc: {2022: 6.4, 2023: 6.53, 2024: 6.75}
Thay đổi toàn quốc 2022→2024: +0.35 điểm
Mức thay đổi trung bình của 9 môn: +0.35 điểm

             Điểm TB 2022  Điểm TB 2024  Mức thay đổi Chiều thay đổi
Sinh học             5.02          6.28          1.26          Tăng
Ngữ văn              6.51          7.23          0.72          Tăng
Địa lý               6.68          7.19          0.51          Tăng
Tiếng Anh            5.16          5.51          0.35          Tăng
Lịch sử              6.34          6.57          0.23          Tăng
GDCD                 8.03          8.16          0.13          Tăng
Toán                 6.47          6.45         -0.02          Giảm/không tăng
Hóa học              6.70          6.68         -0.02          Giảm/không tăng
Vật lý               6.72          6.67         -0.05          Giảm/không tăng

### Kết quả và trả lời

Điểm trung bình toàn quốc tăng từ **6,40 lên 6,75**, tương đương **+0,35 điểm**. Tuy nhiên, mức thay đổi giữa các môn không đồng đều:

- **Sinh học tăng mạnh nhất: +1,26 điểm**.
- **Ngữ văn tăng +0,72 điểm** và **Địa lý tăng +0,51 điểm**.
- **Tiếng Anh tăng +0,35 điểm**, đúng bằng mức tăng của chỉ số toàn quốc.
- Trong khi đó, **Toán, Hóa học và Vật lý không tăng**; cả ba môn đều giảm nhẹ.

### Câu trả lời

**Sự gia tăng điểm trung bình toàn quốc không đến từ sự cải thiện đồng đều của toàn bộ môn thi. Mức tăng tập trung chủ yếu ở Sinh học, Ngữ văn và Địa lý; một số môn khác gần như ổn định hoặc giảm nhẹ.**

Đây là một kết quả có tính cảnh báo: chỉ số toàn quốc tăng nhưng không thể dùng riêng chỉ số này để kết luận rằng mọi môn đều cải thiện. Khi phân tích xu hướng, cần đặt đường toàn quốc cạnh đường của từng môn.

## Câu hỏi 2 — Chênh lệch vùng miền đến từ toàn bộ địa phương hay một số tỉnh, thành?

### Câu hỏi nghiên cứu

Khi một vùng có điểm trung bình cao, kết quả đó có phản ánh mặt bằng chung của các tỉnh, thành trong vùng hay bị chi phối bởi một số địa phương dẫn đầu?

### Vì sao câu hỏi này quan trọng?

Điểm trung bình vùng có thể che khuất sự phân tán nội vùng. Hai vùng có điểm trung bình gần nhau nhưng có thể khác nhau hoàn toàn: một vùng có các tỉnh tương đối đồng đều, vùng kia có khoảng cách rất lớn giữa tỉnh cao nhất và thấp nhất.

### Cách phân tích

1. Sử dụng bộ lọc mặc định của tab **Địa phương & Vùng miền**: năm 2026, môn Tổng hợp, toàn quốc.
2. So sánh biểu đồ **Điểm TB theo vùng miền** với biểu đồ **Top/Bottom tỉnh, thành theo điểm TB**.
3. Tính cho mỗi vùng: điểm TB vùng, tỉnh cao nhất, tỉnh thấp nhất, khoảng cách nội vùng và mức cao hơn của tỉnh dẫn đầu so với điểm TB vùng.
4. Đọc kết quả cùng bảng xếp hạng để phân biệt vùng có mặt bằng đồng đều với vùng có phân hóa nội vùng lớn.

In [ ]:
# Phân tích câu hỏi 2: tái tạo đúng logic mặc định của RegionTab.
# Bộ lọc: năm 2026, môn Tổng hợp, toàn quốc.
subjects_2026 = {
    subject["id"] for subject in app_data["SUBJECTS"]
    if "CT2018" in subject.get("programs", [])
}

province_rows = pd.DataFrame(app_data["provinceRankings"])
province_rows = province_rows[
    province_rows["year"].eq(2026)
    & province_rows["subjectId"].isin(subjects_2026)
].copy()

# RegionTab lấy trung bình các điểm TB theo môn của từng tỉnh.
province_aggregate = (
    province_rows
    .groupby(["province", "regionId", "regionName"], as_index=False)
    .agg(**{"Điểm TB tỉnh": ("average", "mean")})
)

region_rows = pd.DataFrame(app_data["regionAverages"])
region_rows = region_rows[
    region_rows["year"].eq(2026)
    & region_rows["subjectId"].isin(subjects_2026)
].copy()
region_average = (
    region_rows
    .groupby(["regionId", "regionName"], as_index=False)
    .agg(**{"Điểm TB vùng": ("average", "mean")})
)

rows = []
for _, region in region_average.iterrows():
    local = province_aggregate[province_aggregate["regionId"].eq(region["regionId"])].copy()
    local = local.sort_values("Điểm TB tỉnh", ascending=False)
    top = local.iloc[0]
    bottom = local.iloc[-1]
    rows.append({
        "Vùng": region["regionName"],
        "Điểm TB vùng": region["Điểm TB vùng"],
        "Tỉnh cao nhất": top["province"],
        "Điểm cao nhất": top["Điểm TB tỉnh"],
        "Tỉnh thấp nhất": bottom["province"],
        "Điểm thấp nhất": bottom["Điểm TB tỉnh"],
        "Khoảng cách nội vùng": top["Điểm TB tỉnh"] - bottom["Điểm TB tỉnh"],
        "Lợi thế của tỉnh dẫn đầu": top["Điểm TB tỉnh"] - region["Điểm TB vùng"],
    })

region_comparison = pd.DataFrame(rows).sort_values("Điểm TB vùng", ascending=False)
display(region_comparison.round(2))

                                   Vùng  Điểm TB vùng Tỉnh cao nhất  Điểm cao nhất Tỉnh thấp nhất  Điểm thấp nhất  Khoảng cách nội vùng  Lợi thế của tỉnh dẫn đầu
                     Đồng bằng sông Hồng          5.99       Ninh Bình          6.15       Hưng Yên            5.83                 0.32                      0.16
                           Đông Nam Bộ          5.81       Hồ Chí Minh          5.97       Tây Ninh            5.49                 0.48                      0.16
Bắc Trung Bộ và Duyên hải miền Trung          5.75          Hà Tĩnh          6.17       Khánh Hòa            5.49                 0.68                      0.42
           Trung du và miền núi phía Bắc          5.52          Phú Thọ          5.97       Sơn La            5.03                 0.94                      0.45
                              Tây Nguyên          5.48        Lâm Đồng          5.60       Đắk Lắk            5.33                 0.27                      0.12
                 Đồng bằng 

### Kết quả và trả lời

Đồng bằng sông Hồng có điểm TB vùng cao nhất (**5,99 điểm**), nhưng khoảng cách giữa tỉnh cao nhất và thấp nhất chỉ **0,32 điểm**; tỉnh dẫn đầu cao hơn mức vùng **0,16 điểm**. Điều này cho thấy kết quả cao của vùng tương đối đồng đều, không chỉ do một tỉnh đơn lẻ.

Ngược lại, Trung du và miền núi phía Bắc có điểm TB vùng **5,52 điểm** nhưng khoảng cách nội vùng lên tới **0,94 điểm**: Phú Thọ đạt **5,97**, trong khi Sơn La đạt **5,03**. Bắc Trung Bộ và Duyên hải miền Trung cũng có khoảng cách nội vùng khá lớn (**0,68 điểm**), với lợi thế **0,42 điểm** của Hà Tĩnh so với mức trung bình vùng.

### Câu trả lời

**Chênh lệch vùng miền không có một cơ chế duy nhất. Một số vùng có mặt bằng tương đối đồng đều, trong khi một số vùng khác chịu ảnh hưởng rõ hơn từ khoảng cách giữa các tỉnh dẫn đầu và các tỉnh còn lại.**

Vì vậy, không nên suy luận rằng điểm trung bình cao của một vùng luôn phản ánh đồng đều toàn bộ địa phương trong vùng. Cần đọc đồng thời điểm TB vùng, Top/Bottom tỉnh và độ rộng khoảng cách nội vùng.

## Kết luận và giới hạn

### Hai phát hiện chính

1. Điểm trung bình toàn quốc tăng trong giai đoạn 2022–2024, nhưng mức tăng tập trung ở một số môn; các môn còn lại không cải thiện đồng đều.
2. Điểm trung bình vùng có thể che khuất sự khác biệt giữa các tỉnh, thành. Có vùng tương đối đồng đều và có vùng phân hóa nội vùng rõ rệt.

### Giới hạn diễn giải

- Câu hỏi 1 sử dụng giai đoạn CT2006 để bảo đảm tính so sánh giữa các năm; không suy rộng kết quả này cho giai đoạn CT2018.
- Câu hỏi 2 sử dụng bộ lọc năm 2026 và môn Tổng hợp theo đúng trạng thái mặc định của tab Địa phương & Vùng miền; kết luận có thể thay đổi khi chọn năm hoặc môn khác.
- Các chỉ số cho biết mức độ và cấu trúc khác biệt, nhưng không xác định nguyên nhân của chênh lệch giữa môn học, vùng hoặc tỉnh, thành.